# Simulated Data
- The perpetual problem with this project is the lack of good data :<
- Can try mitigate this with some simulated data.
---
- Start with a distribution of star types. 
- Draw a star type from this distribution.
- Draw Prot and Pcyc from distributions defined in SM2016.
- Draw Plong between 50 and 150 years randomly.
- From solar data {https://link.springer.com/article/10.12942/lrsp-2010-1}, the sunspot count varies by roughly 2 times from peak to peak. Plong envelope can then be stochastic about this.
- The amplitudes are a matter of scientific uncertainty.
    - SM2016 Fig 19 implies shorter rotation periods go with larger cycle amplitudes.
    - This contradicts Saar and Brandenburg 2002.
    - May just be easier to draw from a KDE of Fig 22 in SM2016.
- Noise is difficult to do. Just use same error_percent on each point as the magnitude of a Gaussian spread.


In [1]:
import numpy as np

### Star type dist
- We only have data for FGKM stars.
- {https://ui.adsabs.harvard.edu/abs/2001JRASC..95...32L/abstract} has the probabilities of each.
    - F: 3
    - G: 7.6
    - K: 12
    - M: 76 {https://iopscience.iop.org/article/10.3847/1538-4365/abd7a8}
        - Early M: 95 thereof
        - Mid M: 5 thereof

In [39]:
probs_unscaled = [0.03, 0.076, 0.12, 0.76*0.95, 0.76*0.05]
labels = np.array(["F", "G", "K", "ME", "MM"])
probs = probs_unscaled/np.sum(probs_unscaled)
star_type = np.random.choice(len(probs), p=probs, size=100)

In [50]:
from scipy.stats import truncnorm

P_cyc_lookup = np.array([[9.5, 5.3],[6.7,3.6],[8.5,3.6],[6.0,2.9],[7.1,2.7]]) # FGKMEMM, Years, mu sigma
P_rot_lookup = np.array([[8.6, 6.2],[19.6,11.1],[27.4,15.7],[36.2,29.9],[85.4,53.4]]) # Days

P_cyc_samp = P_cyc_lookup[star_type]
P_rot_samp = P_rot_lookup[star_type]

# Use truncated normal to not have negative periods
a_cyc = -P_cyc_samp[:,0]/ P_cyc_samp[:,1]  # lower bound in standardised units
a_rot = -P_rot_samp[:,0]/ P_rot_samp[:,1] 

P_cycs = truncnorm.rvs(a_cyc, np.inf, loc=P_cyc_samp[:,0], scale=P_cyc_samp[:,1])
P_rots = truncnorm.rvs(a_rot, np.inf, loc=P_rot_samp[:,0], scale=P_rot_samp[:,1])
